## Práctica: word2vec / skip-gram

In [29]:
with open("dataset_word2vec.txt", encoding="utf-8") as f:
    frases = [linea.strip() for linea in f if linea.strip()]

print(len(frases), "frases")
print(frases[:3])

251 frases
['el perro corre en el parque', 'el perro come carne', 'el perro bebe agua']


Entrenar un modelo el cual aprenda nuestro vocabulario, para crear un diccionario el cual tenga la palabra y su id como valor dentro de un diccionario, después crear el mismo diccionario inverso para poder recuperar las palabras

In [30]:
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
tokenizer = Tokenizer(models.WordLevel(unk_token="<unk>"))
tokenizer.normalizer = normalizers.Lowercase()
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

In [31]:
trainer = trainers.WordLevelTrainer(special_tokens=["<unk>"])
tokenizer.train_from_iterator(frases, trainer=trainer)

In [32]:
vocab_size = tokenizer.get_vocab_size()
print("Tamaño del vocabulario:", vocab_size)

voc_dicc = tokenizer.get_vocab()
voc_dicc_inverso = {i: w for w, i in voc_dicc.items()}
print(list(voc_dicc.items())[:10])
print(list(voc_dicc_inverso.items())[:10])

Tamaño del vocabulario: 357
[('voy', 183), ('vuelvo', 184), ('rojo', 162), ('estación', 117), ('tren', 56), ('profesor', 319), ('prepara', 154), ('chef', 65), ('madrid', 133), ('tarde', 82)]
[(183, 'voy'), (184, 'vuelvo'), (162, 'rojo'), (117, 'estación'), (56, 'tren'), (319, 'profesor'), (154, 'prepara'), (65, 'chef'), (133, 'madrid'), (82, 'tarde')]


In [33]:
frases_ids = [tokenizer.encode(f).ids for f in frases]
print(frases[0], "->", frases_ids[0])

el perro corre en el parque -> [2, 13, 22, 5, 2, 35]


In [34]:
# Generar pares (centro, contexto) con ventana deslizante
ventana = 2
pares = []

for ids in frases_ids:
    for i, centro in enumerate(ids):
        for j in range(max(0, i - ventana), min(len(ids), i + ventana + 1)):
            if j != i:
                pares.append((centro, ids[j]))

print(len(pares), "pares")
print([(voc_dicc_inverso[c], voc_dicc_inverso[x]) for c, x in pares[:5]])

3086 pares
[('el', 'perro'), ('el', 'corre'), ('perro', 'el'), ('perro', 'corre'), ('perro', 'en')]


In [35]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

centros = torch.tensor([p[0] for p in pares])
contextos = torch.tensor([p[1] for p in pares])
loader = DataLoader(TensorDataset(centros, contextos), batch_size=64, shuffle=True)

class SkipGram(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim)
        self.out = nn.Linear(dim, vocab_size)

    def forward(self, centro):
        return self.out(self.emb(centro))